In [0]:
%pip install databricks-feature-engineering --quiet

%restart_python

In [0]:

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

In [0]:
catalog_name = "ct_oil_gas" 
gold_schema_name = "sc_gold"
fe_table_name = "feature_demand"
eval_split_table_name = "demand_eval_split"
feature_table_name = 'daily_demand_features'
demand_labels_table_name = "daily_demand_labels"


In [0]:
df_feat = spark.table(f"{catalog_name}.{gold_schema_name}.{fe_table_name}")
df_feat.limit(10).display()

In [0]:
## First split the data based on time

cutoff_date = df_feat.select("transaction_date").distinct().orderBy("transaction_date").collect()

cutoff_date

In [0]:
n = len(cutoff_date)
split_idx = int(n*0.8)
split_idx

In [0]:
cutoff = cutoff_date[split_idx]['transaction_date']
cutoff

In [0]:
feature_table_df = df_feat.withColumn(
    "split",
    F.when(F.col("transaction_date") <F.lit(cutoff),'train').otherwise("test")
)

In [0]:
display(feature_table_df.head(10))

In [0]:
display(feature_table_df.tail(10))

In [0]:
split_df = feature_table_df.select("p_key", "transaction_date", "split")

split_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.{gold_schema_name}.{eval_split_table_name}")

In [0]:
demand_labels_df= feature_table_df.select("p_key", "transaction_date", "total_demand","split")


demand_labels_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.{gold_schema_name}.{demand_labels_table_name}")

In [0]:
feature_only_df = feature_table_df.drop("total_demand","split")

In [0]:
feature_only_df.display()

In [0]:
feature_table_df.select("product_name").distinct().show()

## CREATE A FEATURE STORE AND STORE 

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient


fe = FeatureEngineeringClient()

In [0]:
demand_feature_table = fe.create_table(
    name=f"{catalog_name}.{gold_schema_name}.{feature_table_name}",
    primary_keys=["p_key","transaction_date"],
    df = feature_only_df,
    description="Demand forecasting features for product-city pairs, daily grain",
    schema=feature_only_df.schema,
    tags={
        "domain": "oil_gas_demand_forecasting",
        "grain": "product_city_daily",
        "use_case": "time_series_forecasting",
        "source": "sc_gold.feature_demand",
        "quality": "gold",
        },
    timeseries_column="transaction_date"
)

In [0]:
fe.write_table(
    name=f"{catalog_name}.{gold_schema_name}.{feature_table_name}",
  df=feature_only_df, 
  mode='merge' 
  )